# Prueba real: Forex H1 parquet + CUDA env

Este notebook usa los parquet H1 reales, alinea multiples pares por timestamp, calcula log returns y crea episodios paralelos en VRAM para `CuPortfolioEnv`.

In [ ]:
from pathlib import Path
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'pruebas_reales':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from forex_cuda import CuPortfolioEnv, PortfolioEnvConfig
from forex_cuda.preprocessing import build_or_load_preprocessed_dataset
from forex_cuda.tensors import make_rolling_market_tensor

print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
DATA_DIR = Path(os.environ.get('FOREX_CUDA_DATA_DIR', PROJECT_ROOT / 'data' / 'parquet_h1'))
CACHE_DIR = PROJECT_ROOT / 'data_cache'

PAIRS = [
    'EURUSD',
    'GBPUSD',
    'USDJPY',
    'USDCHF',
    'USDCAD',
    'AUDUSD',
    'NZDUSD',
    'EURJPY',
]

NUM_ENVS = 4096
EPISODE_STEPS = 512
MAX_ROWS = 80_000

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Dataset directory not found: {DATA_DIR}. Set FOREX_CUDA_DATA_DIR=/path/to/parquet_h1'
    )

dataset = build_or_load_preprocessed_dataset(
    DATA_DIR,
    pairs=PAIRS,
    max_rows=MAX_ROWS,
    drop_data_gaps=True,
    cache_dir=CACHE_DIR,
    cache_name='h1_major8',
)

print('pairs:', dataset.pairs)
print('close_prices:', dataset.close_prices.shape)
print('log_returns [assets, time]:', dataset.log_returns.shape)
print('date range:', dataset.timestamps[0], '->', dataset.timestamps[-1])
print('loaded_from_cache:', dataset.metadata.get('loaded_from_cache'))
print('cache_root:', dataset.metadata.get('cache_root'))

In [ ]:
generator = torch.Generator(device='cuda').manual_seed(123)

market = make_rolling_market_tensor(
    dataset.log_returns,
    num_envs=NUM_ENVS,
    episode_steps=EPISODE_STEPS,
    random_starts=True,
    device='cuda',
    generator=generator,
)

print('market:', tuple(market.shape), market.dtype, market.device)
print('contiguous:', market.is_contiguous())

In [ ]:
config = PortfolioEnvConfig(
    initial_balance=10_000.0,
    commission_bps_range=(0.5, 2.0),
    spread_bps_range=(0.0, 0.8),
    slippage_bps_range=(0.0, 0.4),
    max_abs_weight=1.0,
    max_leverage=1.0,
)

env = CuPortfolioEnv(market, config=config)
obs, info = env.reset(seed=123)

print('obs:', tuple(obs.shape), obs.dtype, obs.device)
print('info:', info)
print('mean total cost bps:', float(env.costs.total_bps().mean().item()))

In [ ]:
RUN_STEPS = 256
balance_history = []
reward_history = []

torch.cuda.synchronize()
started = time.perf_counter()

for _ in range(RUN_STEPS):
    actions = torch.empty(NUM_ENVS, len(PAIRS), device='cuda').uniform_(-0.25, 0.25, generator=generator)
    obs, rewards, terminated, truncated, info = env.step(actions)
    balance_history.append(float(env.state.balances.mean().item()))
    reward_history.append(float(rewards.mean().item()))
    if bool(terminated.all().item()):
        break

torch.cuda.synchronize()
elapsed = time.perf_counter() - started

print('steps:', len(balance_history))
print('elapsed seconds:', elapsed)
print('env-steps/sec:', NUM_ENVS * len(balance_history) / elapsed)
print('mean balance:', balance_history[-1])
print('mean reward last:', reward_history[-1])
print('terminated all:', bool(terminated.all().item()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(balance_history)
axes[0].set_title('Mean balance')
axes[0].set_xlabel('step')
axes[0].set_ylabel('equity')

axes[1].plot(reward_history)
axes[1].set_title('Mean reward')
axes[1].set_xlabel('step')

fig.tight_layout()

## Comparacion CPU vs GPU

Este benchmark usa las mismas ventanas reales H1, acciones y costos por entorno para comparar un loop tradicional Python/Numpy contra el kernel CUDA vectorizado.

In [ ]:
!python {PROJECT_ROOT / 'benchmarks' / 'benchmark_real_parquet.py'}

import json

real_benchmark_path = PROJECT_ROOT / 'benchmark_results' / 'benchmark_real_parquet.json'
real_benchmark = json.loads(real_benchmark_path.read_text())

print('CPU seconds:', real_benchmark['cpu_python_numpy']['seconds'])
print('GPU seconds:', real_benchmark['gpu_cuda_vectorized']['seconds'])
print('speedup:', real_benchmark['speedup'])

## Fine-tuning con costos fijos

Para simular condiciones reales medidas de un broker/API, congela los costos antes de entrenar con learning rate bajo.

In [ ]:
env.set_costs(
    commission_bps=0.8,
    spread_bps=0.35,
    slippage_bps=0.15,
)

print('fixed total cost bps:', float(env.costs.total_bps().mean().item()))